In [1]:
## Import Libraries

In [29]:
import pandas as pd

In [30]:
## load the dataset

In [31]:
df_adult = pd.read_csv("data/adult.csv") 
print("shape of dataset is:",df_adult.shape)
print(df_adult.head())

shape of dataset is: (32561, 15)
   age          workclass  fnlwgt   education  education-num  \
0   39          State-gov   77516   Bachelors             13   
1   50   Self-emp-not-inc   83311   Bachelors             13   
2   38            Private  215646     HS-grad              9   
3   53            Private  234721        11th              7   
4   28            Private  338409   Bachelors             13   

        marital-status          occupation    relationship    race      sex  \
0        Never-married        Adm-clerical   Not-in-family   White     Male   
1   Married-civ-spouse     Exec-managerial         Husband   White     Male   
2             Divorced   Handlers-cleaners   Not-in-family   White     Male   
3   Married-civ-spouse   Handlers-cleaners         Husband   Black     Male   
4   Married-civ-spouse      Prof-specialty            Wife   Black   Female   

   capital-gain  capital-loss  hours-per-week  native-country  target  
0          2174             0      

In [32]:
## Data Exploration EDA

In [33]:
# Info Dataset
print(df_adult.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education-num   32561 non-null  int64 
 5   marital-status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital-gain    32561 non-null  int64 
 11  capital-loss    32561 non-null  int64 
 12  hours-per-week  32561 non-null  int64 
 13  native-country  32561 non-null  object
 14  target          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB
None


In [34]:
# summary statistics
print(df_adult.describe())

                age        fnlwgt  education-num  capital-gain  capital-loss  \
count  32561.000000  3.256100e+04   32561.000000  32561.000000  32561.000000   
mean      38.581647  1.897784e+05      10.080679   1077.648844     87.303830   
std       13.640433  1.055500e+05       2.572720   7385.292085    402.960219   
min       17.000000  1.228500e+04       1.000000      0.000000      0.000000   
25%       28.000000  1.178270e+05       9.000000      0.000000      0.000000   
50%       37.000000  1.783560e+05      10.000000      0.000000      0.000000   
75%       48.000000  2.370510e+05      12.000000      0.000000      0.000000   
max       90.000000  1.484705e+06      16.000000  99999.000000   4356.000000   

       hours-per-week  
count    32561.000000  
mean        40.437456  
std         12.347429  
min          1.000000  
25%         40.000000  
50%         40.000000  
75%         45.000000  
max         99.000000  


In [35]:
# To check Missing or NaN value 

In [36]:
print(df_adult.isna().sum())

age               0
workclass         0
fnlwgt            0
education         0
education-num     0
marital-status    0
occupation        0
relationship      0
race              0
sex               0
capital-gain      0
capital-loss      0
hours-per-week    0
native-country    0
target            0
dtype: int64


In [37]:
# Duplicates 

In [38]:
print("Duplicates:",df_adult.duplicated().sum())

Duplicates: 24


In [42]:
# Target variable distribution
print("distribution",df_adult['target'].value_counts(normalize=True))
print("distribution------‚",df_adult['workclass'].value_counts(normalize=True))


distribution target
<=50K    0.75919
>50K     0.24081
Name: proportion, dtype: float64
distribution------‚ workclass
Private             0.697030
Self-emp-not-inc    0.078038
Local-gov           0.064279
?                   0.056386
State-gov           0.039864
Self-emp-inc        0.034274
Federal-gov         0.029483
Without-pay         0.000430
Never-worked        0.000215
Name: proportion, dtype: float64


In [43]:
# Handle Missing Value 

In [51]:
## Replace ? with NaN
df_adult.replace("?", pd.NA, inplace=True)
#check again
print(df_adult.isna().sum())
# Drop rows with missing values 
df_adult = df_adult.dropna()

age               0
workclass         0
fnlwgt            0
education         0
education-num     0
marital-status    0
occupation        0
relationship      0
race              0
sex               0
capital-gain      0
capital-loss      0
hours-per-week    0
native-country    0
target            0
dtype: int64


In [52]:
## Feature Engineering for columns or feature

In [65]:
df_adult['age_group'] = pd.cut(df_adult["age"], bins=[0, 25, 35, 45, 55, 65, 75, 100], labels=["<25", "25-35","35-45","45-55","55-65", "65-75","75+"])

In [66]:
# Split into Train and Test sets

In [69]:
from sklearn.model_selection import train_test_split
X = df_adult.drop(columns=["target"])
y = df_adult['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)


Train Shape: (26048, 15)
Test Shape: (6513, 15)


In [70]:
## Preprocessing  Encoding and Scaling

In [84]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression


cate_cols = X_train.select_dtypes(include="object").columns
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns

cate_transformer = OneHotEncoder(handle_unknown="ignore")
num_transformer = StandardScaler()

preprocessor = ColumnTransformer(
    transformers=[
      ("num", num_transformer,num_cols),
      ("cat", cate_transformer,cate_cols),
    ]
)

clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])



In [94]:
## Train Model
clf.fit(X_train, y_train)

print("Train accuracy:----",clf.score(X_train,y_train)*100)
print("Test accuracy:----",clf.score(X_test,y_test)*100)


Train accuracy:---- 85.20423832923832
Test accuracy:---- 85.5980346998311


In [95]:
##Model Evaluation

In [101]:
from sklearn.metrics import classification_report, confusion_matrix
y_pred = clf.predict(X_test)
print("Classification Report------", classification_report(y_test,y_pred))
print("Confusion matrix ------", confusion_matrix(y_test,y_pred))

Classification Report------               precision    recall  f1-score   support

       <=50K       0.89      0.93      0.91      4945
        >50K       0.74      0.62      0.67      1568

    accuracy                           0.86      6513
   macro avg       0.81      0.78      0.79      6513
weighted avg       0.85      0.86      0.85      6513

Confusion matrix ------ [[4603  342]
 [ 596  972]]
